# Value Strategy — B/M ajusté intangibles · Qualité · Momentum · Détection ML de régimes

Notebook **narratif** : il importe la logique depuis le package `value_strategy` (dans `src/`) 
et raconte la stratégie partie par partie. Tout le code lourd vit dans les modules ; ce notebook 
ne fait qu'orchestrer et commenter.

**Stratégie** : long/short equity US Small/Mid Cap. Signal value = B/M ajusté intangibles 
(KC + OC, Peters & Taylor 2017), ranké par secteur, filtré qualité + momentum 6M. Short = bottom 
qualité du bucket growth. Détection ML des régimes *junk rally* pour réduire dynamiquement le short.

> **Données** : CRSP + Compustat via WRDS · **IS** 2003-2013 · **OOS** 2014-2024  
> Première exécution : connexion WRDS (met les données en cache). Ensuite : `USE_CACHE = True`.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Rend le package importable depuis le notebook (src/ sur le path)
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from value_strategy import config, signals, factors, dynamic_short, plots, reports
from value_strategy import portfolio as pf
from value_strategy.data import build as data_build
from value_strategy.ml_regime import pipeline as ml_pipeline

config.ensure_dirs()
USE_CACHE = True   # rejoue depuis le cache local data/cache/ (rapide, sans WRDS) ; mettre False pour repartir de WRDS

## Partie 1 — Préparation des données

Connexion WRDS, extraction CRSP + Compustat, correction du survivorship bias (Shumway 2001), 
calcul des capitaux immatériels KC/OC, nettoyage, illiquidité d'Amihud et coûts dynamiques. 
Le panel mensuel final est mis en cache (parquet) pour la reproductibilité.

In [2]:
if USE_CACHE:
    panel, df_comp, crsp_ml, macro = data_build.load_cached_panel()
else:
    panel, df_comp, crsp_ml, macro = data_build.build_panel_from_wrds()
panel.shape

  PARTIE 1 — LECTURE DU PANEL (cache)


  → cache lu : data/cache/panel.parquet (1,191,883 lignes)
  → cache lu : data/cache/compustat.parquet (210,222 lignes)
  → cache lu : data/cache/crsp_ml.parquet (1,294,973 lignes)


  → cache lu : data/cache/macro.parquet (300 lignes)


(1191883, 50)

## Partie 2 — In-sample : construction et calibration (2003-2013)

Signaux value (neutralisés par secteur) + score qualité + momentum, puis construction des 
portefeuilles long/short (rebalancement semestriel) et performance nette de tous les coûts.

In [3]:
panel_sm_is, filtered_is = signals.build_signals(panel, config.IS_START, config.IS_END)
long_is, short_is, _ = pf.construct_portfolios(filtered_is, config.IS_START, config.IS_END)
perf_is, costs_is = pf.compute_performance(panel_sm_is, long_is, short_is, config.IS_START, config.IS_END)

Panel 2003-2013 : 549,723 obs — 7,063 titres


Univers investissable : 4,196 titres uniques


Univers filtré : 3,359 titres


Nb dates rebalancement : 22
Taille moyenne LONG  : 84.0
Taille moyenne SHORT : 130.8



Performance calculée : 120 mois
Coût transaction : 0.27%/an
Coût borrow      : 1.26%/an
Coût total       : 1.53%/an


## Partie 3 — Out-of-sample : application et comparaison (2014-2024)

**Mêmes paramètres, aucun recalibrage.** On télécharge les facteurs Fama-French et on compare 
IS vs OOS (Sharpe, alpha FF4, information ratio vs HML).

In [4]:
panel_sm_oos, filtered_oos = signals.build_signals(panel, config.OOS_START, config.OOS_END)
long_oos, short_oos, _ = pf.construct_portfolios(filtered_oos, config.OOS_START, config.OOS_END)
perf_oos, costs_oos = pf.compute_performance(panel_sm_oos, long_oos, short_oos, config.OOS_START, config.OOS_END)

ff5, mom_ff = factors.load_ff_factors()
stats_is, stats_oos = reports.print_is_oos_comparison(perf_is, perf_oos, ff5, mom_ff, costs_is, costs_oos)

Panel 2014-2024 : 446,929 obs — 5,724 titres


Univers investissable : 3,964 titres uniques


Univers filtré : 2,976 titres


Nb dates rebalancement : 22
Taille moyenne LONG  : 81.3
Taille moyenne SHORT : 102.0



Performance calculée : 120 mois
Coût transaction : 0.34%/an
Coût borrow      : 0.86%/an
Coût total       : 1.20%/an


Facteurs FF chargés avec succès

COMPARAISON IS vs OOS — Stratégie nette (après tous coûts)

Métrique                       IS (2003-13)   OOS (2014-24)
-----------------------------------------------------------
Rendement annuel                     12.01%          16.01%
Volatilité                           10.66%          11.30%
Sharpe Ratio                           0.92            1.18
Max Drawdown                        -11.99%         -12.83%
Calmar Ratio                           1.00            1.25
Skewness                               0.94            0.78
VaR 5%                               -3.17%          -3.18%
-----------------------------------------------------------
Alpha FF4 annuel                     12.00%          15.71%
t-stat alpha FF4                       3.52            4.22
R² FF4                                0.036           0.026
Info. Ratio vs HML                     1.09            1.34

  → Variation Sharpe IS→OOS : +0.25
  → Coût IS : 1.53%/an | Coût 

## Partie 4 — Détection ML de régime « junk rally »

Feature engineering (CRSP + Compustat + FRED) → labeling HMM/GMM des régimes d'euphorie 
(VIX bas) → prédiction supervisée walk-forward → signaux `SHORT_REDUCE` / `FULL_SHORT`.

In [5]:
ml = ml_pipeline.run_ml_regime(crsp_ml, df_comp, macro)
ml['metrics_df']

  PARTIE 4 — DÉTECTION DE RÉGIME (ML)

[STEP 1] Feature engineering V3...


  Dropped 6,407 lignes sans ret


  Panel marché brut : 300 mois (2000-01 → 2024-12)
  Merged 6 séries macro
  Panel marché final : 300 mois | Features : 56
[STEP 2] Labeling régimes EUPHORIA V3 (min_dur=3, bridge=2)...


  ⚠ Clustering dégénéré (régime minoritaire 6.0% < 20%) → fallback score de stress (médiane)
  Méthode : GaussianHMM + fallback score-stress (clustering dégénéré)
  Normal   (0) :  131 mois (43.7%)
  Euphoria (1) :  169 mois (56.3%)
  Features : ['vix_z', 'rvol_mkt_z', 'ret_dispersion_z', 'credit_spread_z']
  VIX moyen en Normal   : -0.82
  VIX moyen en Euphoria : +0.53
[STEP 3a] Préparation données supervisées...
  Samples : 299 | Stress : 56.2% | Période : 2000-01 → 2024-11
[STEP 3b] Walk-forward V3 (min_train=120, test=12, n=299)...


  Folds : 14 | OOS : 168 | Seuil moyen : 0.472

  COMPARAISON MODÈLES — Out-of-Sample V3

  Logistic Regression:
    Accuracy: 0.774 | Precision: 0.840 | Recall: 0.792
    AUC: 0.784 | Brier: 0.1960
    Confusion matrix: [[46, 16], [22, 84]]

  Gradient Boosting:
    Accuracy: 0.774 | Precision: 0.840 | Recall: 0.792
    AUC: 0.821 | Brier: 0.1751
    Confusion matrix: [[46, 16], [22, 84]]

  ★ Meilleur (AUC) : Gradient Boosting (0.821)

[STEP 4b] Feature importance (permutation)...


  Top 10 features :
    vix_z                               0.0369 ± 0.0102
    turnover_mkt_z                      0.0176 ± 0.0072
    leverage_mkt_z                      0.0169 ± 0.0078
    ret_dispersion_z                    0.0129 ± 0.0080
    log_dollar_vol_z                    0.0107 ± 0.0075
    vix_z_ma3                           0.0097 ± 0.0051
    consumer_sent_z_chg3                0.0086 ± 0.0055
    ted_spread_z_ma3                    0.0075 ± 0.0033
    fed_funds_z_chg1                    0.0074 ± 0.0052
    mkt_vol_3m                          0.0062 ± 0.0021
[STEP 6] Signaux euphoria (seuil=0.472)...
  SHORT_REDUCE : 101 | FULL_SHORT : 67
  Euphoria détectée : 85/106 (80.2%)
  Précision globale signal GB : 78.0%

  RÉSUMÉ PARTIE 4 — DÉTECTION EUPHORIA
  Méthode régime   : GaussianHMM + fallback score-stress (clustering dégénéré) + smoothing
  Features totales : 56
  Panel ML         : 300 mois
  Seuil moyen      : 0.472


,Accuracy,Precision (stress),Recall (stress),F1 (stress),AUC-ROC,Brier Score
Model,,,,,,
Logistic Regression,0.77381,0.84,0.792453,0.815534,0.784236,0.196038
Gradient Boosting,0.77381,0.84,0.792453,0.815534,0.820907,0.175073


## Partie 5 — Réduction dynamique du short en junk rally

Quand le ML détecte l'euphorie, le poids du short passe de 100 % à 50 %. On compare la 
stratégie originale et la version *short dynamique* (Sharpe, MaxDD, alpha FF4, coûts).

In [6]:
ff = factors.download_ff_factors()
perf_h, summary = dynamic_short.build_dynamic_short(perf_oos, ml['signals_df'], costs_oos)
perf_h, metrics = dynamic_short.attach_ff_and_metrics(perf_h, ff)
ff4_results = dynamic_short.ff4_regression(perf_h, ff)
reports.print_dynamic_short_verdict(perf_h, metrics, ff4_results, summary, costs_oos)

  PARTIE 5 — SHORT DYNAMIQUE PILOTÉ PAR LE ML
[5.1b] Alignement signaux ML → rendements OOS...

  Période : 2014-12 → 2024-11 (120 mois)
  Mois SHORT_REDUCE (50%) : 85 (71%)
  Mois FULL_SHORT (100%)  : 35 (29%)
  Transitions             : 14
  Coût transitions total  : 2.10%

  RÉSUMÉ FINAL — FAUT-IL GARDER LE SHORT DYNAMIQUE ?
  Sharpe  : 1.177 → 1.201 (+0.024)
  CAGR    : 15.3% → 19.7% (+4.4%)
  MaxDD   : -12.8% → -11.2% (+1.6%)
  α FF4   : 0.1571 → 0.2060
  Mois short réduit (50%) : 85/120 (71%) | Transitions : 14
  Coût annualisé : 1.20% → 1.04%

  VERDICT : ✓ Sharpe AMÉLIORÉ ET MaxDD réduit → GARDER


## Figures

Toutes les figures sont sauvegardées dans `results/charts/`.

In [7]:
plots.setup_style()
plots.plot_cumulative_wealth(perf_is, perf_oos, ff5)
plots.plot_rolling_sharpe(perf_is, perf_oos, ff5)
plots.plot_annual_returns(perf_is, perf_oos, ff5)
plots.plot_drawdown(perf_is, perf_oos, ff5)
plots.plot_calendar_heatmap(perf_oos)
plots.plot_robustness(perf_is, perf_oos, ff5)
plots.plot_ml_dashboard(ml['mkt_df'], ml['results_df'], ml['importance_df'], ml['metrics_df'], ml['regime_method'], ml['avg_threshold'])
plots.plot_dynamic_short(perf_h, metrics, ff4_results, summary)
plots.plot_final_comparison(perf_is, perf_oos, perf_h, ff5, stats_is, stats_oos, metrics['adj'])

  ✓ figure : results/charts/fig1_cumulative_wealth.png
  ✓ figure : results/charts/fig2_rolling_sharpe.png


  ✓ figure : results/charts/fig3_annual_returns.png
  ✓ figure : results/charts/fig4_drawdown.png


  ✓ figure : results/charts/fig5_calendar_heatmap.png
  ✓ figure : results/charts/fig6_robustesse.png


  ✓ figure : results/charts/liquidity_regime_dashboard_v3.png


  ✓ figure : results/charts/short_dynamique_v3.png


  ✓ figure : results/charts/fig_final_cumulative_comparison.png
  Valeur finale $1 : originale $12.22 | short dyn. $17.71 | HML $0.82


PosixPath('/home/lucad/Documents/Projets Programmation/Value-Strategy/results/charts/fig_final_cumulative_comparison.png')